# Check quality of Ethica elite files

A very rough check of the accelerometer and GPS data captured by Ethica shows that some temporal gaps might be present between the two streams for a same participant/wave, which should not be the case. Givent the many issues that marred the Ethica sensor data collection, particularly in the first waves, a rapid assessment of the overlaps and gaps between the AXL and GPS data is done.

## Setup

_NB_ create a personalized Jupyter kernel:

```sh
module load StdEnv/2023 python/3.11 scipy-stack/2026a arrow/25.0.0
virtualenv --no-download $HOME/jpy_env
source $HOME/jpy_env/bin/activate
pip install --no-index --upgrade pip
pip install --no-index tabulate SQLAlchemy psycopg2 polars
pip install --no-index ipykernel jupyterlab
ipython kernel install --user --name=jpy_env --display-name="Python (my_jpy_env)"
```

Then, when running in JupyterLab, one needs to load the required modules in the toolbar (left) beforehand.

In [1]:
import os
import re
import pandas as pd
import polars as pl
from tqdm import tqdm

## Read data

We extract the min and max timestamps of the AXL and GPS data streams.

In [2]:
data_folder = r'/home/btcrchum/projects/def-dfuller/interact/data_archive'

# Define city_id and wave_id
cities = {'mtl': 'montreal', 
            'skt': 'saskatoon', 
            'van': 'vancouver', 
            'vic': 'victoria'}
waves = [1, 2, 3, 4]

# List data files in elite subfolders
n_files = []
axl_files = []
gps_files = []

for ccode, city in cities.items():
    for wave in waves:
        curdir = os.path.join(data_folder, city, f'wave_{wave:02d}', 'ethica_elite_files')
        files = os.listdir(curdir)
        _axl_files = [(city, wave, f) for f in files if re.match('\\d+_AXL.csv', f)]
        _gps_files = [(city, wave, f) for f in files if re.match('\\d+_GPS.csv', f)]
        n_files.append((city, wave, len(_axl_files), len(_gps_files)))
        axl_files += _axl_files
        gps_files += _gps_files

print(pd.DataFrame.from_records(n_files, columns=['City', 'Wave', 'AXL files', 'GPS files']))

         City  Wave  AXL files  GPS files
0    montreal     1        556         94
1    montreal     2        164        162
2    montreal     3        162        161
3    montreal     4        189        186
4   saskatoon     1        152         15
5   saskatoon     2         80         77
6   saskatoon     3         80         77
7   saskatoon     4         41         41
8   vancouver     1        132        131
9   vancouver     2         92         92
10  vancouver     3         77         75
11  vancouver     4         85         82
12   victoria     1        150        149
13   victoria     2        139        139
14   victoria     3        119        116
15   victoria     4         81         81


_NB_ Many GPS files are missing from Montréal and Saskatoon, wave 1. Unpacking the archive `interact/from_ethica_W1_W2_W3/all_studies_gps.tar.gz` does not solve the issue as the zipped GPS files are exactly the same as the ones in `interact/data_archive`. Could have been a problem with the surevy configuration or an error while downloading the GPS sensor data from Ethica platform.

## Find time span of AXL and GPS data

In [3]:
%%time

axltspan_list = []

# Extract time span of AXL files
# for city, wave, f in tqdm(_axl_files[:30]): # DEBUG
for city, wave, f in axl_files:
    axlf = os.path.join(data_folder, city, f'wave_{wave:02d}', 'ethica_elite_files', f)
    q = (
        pl.scan_csv(axlf, try_parse_dates=True)
        .select('interact_id', 'record_time')
        .group_by('interact_id')
        .agg(axl_tmin = pl.min('record_time'),
             axl_tmax = pl.max('record_time'))
    )
    df = q.collect()
    df.insert_column(0, pl.lit(city).alias('city'))
    df.insert_column(1, pl.lit(wave).alias('wave'))
    axltspan_list.append(df)

axltspan_df = pl.concat(axltspan_list)
print(axltspan_df)

shape: (2_299, 5)
┌──────────┬──────┬─────────────┬─────────────────────────────┬─────────────────────────────┐
│ city     ┆ wave ┆ interact_id ┆ axl_tmin                    ┆ axl_tmax                    │
│ ---      ┆ ---  ┆ ---         ┆ ---                         ┆ ---                         │
│ str      ┆ i32  ┆ i64         ┆ datetime[μs, UTC]           ┆ datetime[μs, UTC]           │
╞══════════╪══════╪═════════════╪═════════════════════════════╪═════════════════════════════╡
│ montreal ┆ 1    ┆ 401416333   ┆ 2018-08-02 01:50:08.809 UTC ┆ 2018-08-18 04:21:01.590 UTC │
│ montreal ┆ 1    ┆ 401377103   ┆ 2018-09-12 22:26:11.828 UTC ┆ 2018-10-12 22:23:56.467 UTC │
│ montreal ┆ 1    ┆ 401077819   ┆ 2018-08-17 10:14:35.597 UTC ┆ 2018-09-13 20:48:48.667 UTC │
│ montreal ┆ 1    ┆ 401664309   ┆ 2018-07-31 23:59:35.250 UTC ┆ 2018-07-31 23:59:35.805 UTC │
│ montreal ┆ 1    ┆ 401128981   ┆ 2018-08-31 23:56:32.545 UTC ┆ 2018-08-31 23:56:43.150 UTC │
│ …        ┆ …    ┆ …           ┆ …       

In [4]:
%%time 

gpstspan_list = []

# Extract time span of AXL files
#for city, wave, f in tqdm(_gps_files): # DEBUG
for city, wave, f in gps_files:
    gpsf = os.path.join(data_folder, city, f'wave_{wave:02d}', 'ethica_elite_files', f)
    q = (
        pl.scan_csv(gpsf, try_parse_dates=True)
        .select('interact_id', 'record_time')
        .group_by('interact_id')
        .agg(gps_tmin = pl.min('record_time'),
             gps_tmax = pl.max('record_time'))
    )
    df = q.collect()
    df.insert_column(0, pl.lit(city).alias('city'))
    df.insert_column(1, pl.lit(wave).alias('wave'))
    gpstspan_list.append(df)

gpstspan_df = pl.concat(gpstspan_list)
print(gpstspan_df)

shape: (1_678, 5)
┌──────────┬──────┬─────────────┬─────────────────────────────┬─────────────────────────────┐
│ city     ┆ wave ┆ interact_id ┆ gps_tmin                    ┆ gps_tmax                    │
│ ---      ┆ ---  ┆ ---         ┆ ---                         ┆ ---                         │
│ str      ┆ i32  ┆ i64         ┆ datetime[μs, UTC]           ┆ datetime[μs, UTC]           │
╞══════════╪══════╪═════════════╪═════════════════════════════╪═════════════════════════════╡
│ montreal ┆ 1    ┆ 401851002   ┆ 2018-11-27 00:27:57.123 UTC ┆ 2019-01-09 04:58:21.540 UTC │
│ montreal ┆ 1    ┆ 401556493   ┆ 2018-08-22 03:41:13.042 UTC ┆ 2018-09-21 03:30:52.132 UTC │
│ montreal ┆ 1    ┆ 401018766   ┆ 2019-01-21 23:53:33.134 UTC ┆ 2019-02-20 18:44:56.523 UTC │
│ montreal ┆ 1    ┆ 401689880   ┆ 2018-10-23 19:20:17.423 UTC ┆ 2018-11-22 19:12:57.191 UTC │
│ montreal ┆ 1    ┆ 401149339   ┆ 2018-09-12 14:29:38.739 UTC ┆ 2018-10-13 06:20:19.761 UTC │
│ …        ┆ …    ┆ …           ┆ …       

In [5]:
# Merge AXL and GPS time span
tpsan_df = axltspan_df.join(gpstspan_df, on=['city', 'wave', 'interact_id'], how='full', coalesce=True)
print(tpsan_df)

shape: (2_301, 7)
┌───────────┬──────┬─────────────┬────────────────┬────────────────┬───────────────┬───────────────┐
│ city      ┆ wave ┆ interact_id ┆ axl_tmin       ┆ axl_tmax       ┆ gps_tmin      ┆ gps_tmax      │
│ ---       ┆ ---  ┆ ---         ┆ ---            ┆ ---            ┆ ---           ┆ ---           │
│ str       ┆ i32  ┆ i64         ┆ datetime[μs,   ┆ datetime[μs,   ┆ datetime[μs,  ┆ datetime[μs,  │
│           ┆      ┆             ┆ UTC]           ┆ UTC]           ┆ UTC]          ┆ UTC]          │
╞═══════════╪══════╪═════════════╪════════════════╪════════════════╪═══════════════╪═══════════════╡
│ montreal  ┆ 1    ┆ 401416333   ┆ 2018-08-02     ┆ 2018-08-18     ┆ null          ┆ null          │
│           ┆      ┆             ┆ 01:50:08.809   ┆ 04:21:01.590   ┆               ┆               │
│           ┆      ┆             ┆ UTC            ┆ UTC            ┆               ┆               │
│ montreal  ┆ 1    ┆ 401377103   ┆ 2018-09-12     ┆ 2018-10-12     ┆ 2018

## A few stats on GPS and AXL matches

- How many AXL data without GPS match?
- How many GPS with no AXL match?
- How many AXL and GPS with no overlap, and gap descriptive stats?
- How many AXL and GPS matches, and overlap descriptive stats?

In [6]:
# AXL data with no GPS match
with pl.Config(tbl_rows=-1):
    print(tpsan_df.filter(pl.col('gps_tmin').is_null()).group_by(['city', 'wave']).len('N').sort('N', descending=True))

shape: (12, 3)
┌───────────┬──────┬─────┐
│ city      ┆ wave ┆ N   │
│ ---       ┆ ---  ┆ --- │
│ str       ┆ i32  ┆ u32 │
╞═══════════╪══════╪═════╡
│ montreal  ┆ 1    ┆ 463 │
│ saskatoon ┆ 1    ┆ 137 │
│ saskatoon ┆ 2    ┆ 4   │
│ saskatoon ┆ 3    ┆ 3   │
│ vancouver ┆ 4    ┆ 3   │
│ montreal  ┆ 4    ┆ 3   │
│ victoria  ┆ 3    ┆ 3   │
│ vancouver ┆ 3    ┆ 2   │
│ montreal  ┆ 2    ┆ 2   │
│ vancouver ┆ 1    ┆ 1   │
│ victoria  ┆ 1    ┆ 1   │
│ montreal  ┆ 3    ┆ 1   │
└───────────┴──────┴─────┘


In [7]:
# GPS data with no AXL match
with pl.Config(tbl_rows=-1):
    print(tpsan_df.filter(pl.col('axl_tmin').is_null()).group_by(['city', 'wave']).len('N').sort('N', descending=True))

shape: (2, 3)
┌───────────┬──────┬─────┐
│ city      ┆ wave ┆ N   │
│ ---       ┆ ---  ┆ --- │
│ str       ┆ i32  ┆ u32 │
╞═══════════╪══════╪═════╡
│ montreal  ┆ 1    ┆ 1   │
│ saskatoon ┆ 2    ┆ 1   │
└───────────┴──────┴─────┘


In [8]:
# AXL and GPS with no overlap...
gap_df = tpsan_df.filter((pl.col('axl_tmax') < pl.col('gps_tmin')) | (pl.col('gps_tmax') < pl.col('axl_tmin')))
with pl.Config(tbl_rows=-1):
    print(gap_df.group_by(['city', 'wave']).len('N').sort('N', descending=True))

shape: (6, 3)
┌───────────┬──────┬─────┐
│ city      ┆ wave ┆ N   │
│ ---       ┆ ---  ┆ --- │
│ str       ┆ i32  ┆ u32 │
╞═══════════╪══════╪═════╡
│ saskatoon ┆ 1    ┆ 15  │
│ montreal  ┆ 4    ┆ 4   │
│ victoria  ┆ 1    ┆ 2   │
│ montreal  ┆ 1    ┆ 2   │
│ montreal  ┆ 3    ┆ 2   │
│ montreal  ┆ 2    ┆ 1   │
└───────────┴──────┴─────┘


In [9]:
# ... and gap descriptive stats
gap_df = gap_df.with_columns(
    pl.when(pl.col('axl_tmax') < pl.col('gps_tmin'))
    .then(pl.col('gps_tmin') - pl.col('axl_tmax'))
    .when(pl.col('gps_tmax') < pl.col('axl_tmin'))
    .then(pl.col('axl_tmin') - pl.col('gps_tmax'))
    .alias('gap')
)
with pl.Config(tbl_rows=-1):
    print(gap_df.group_by(['city', 'wave']).agg(
        gap_avg = pl.mean('gap'),
        gap_std = pl.std('gap'),
        gap_min = pl.min('gap'),
        gap_max = pl.max('gap')
    ).sort(['city', 'wave']))

shape: (6, 6)
┌───────────┬──────┬────────────────────┬────────────────────┬────────────────┬────────────────────┐
│ city      ┆ wave ┆ gap_avg            ┆ gap_std            ┆ gap_min        ┆ gap_max            │
│ ---       ┆ ---  ┆ ---                ┆ ---                ┆ ---            ┆ ---                │
│ str       ┆ i32  ┆ duration[μs]       ┆ duration[μs]       ┆ duration[μs]   ┆ duration[μs]       │
╞═══════════╪══════╪════════════════════╪════════════════════╪════════════════╪════════════════════╡
│ montreal  ┆ 1    ┆ 2m 819ms           ┆ 2m 48s 650624µs    ┆ 1s 565ms       ┆ 4m 73ms            │
│ montreal  ┆ 2    ┆ 40s 162ms          ┆ null               ┆ 40s 162ms      ┆ 40s 162ms          │
│ montreal  ┆ 3    ┆ 2m 16s 77500µs     ┆ 1m 53s 235372µs    ┆ 56s 8ms        ┆ 3m 36s 147ms       │
│ montreal  ┆ 4    ┆ 4d 49m 9s 36250µs  ┆ 4d 16h 57m 1s      ┆ 56s 419ms      ┆ 9d 1h 38m 24s      │
│           ┆      ┆                    ┆ 859259µs           ┆               

In [10]:
# AXL and GPS with overlap...
ovlp_df = tpsan_df.filter((pl.col('axl_tmax') >= pl.col('gps_tmin')) & (pl.col('gps_tmax') >= pl.col('axl_tmin')))
with pl.Config(tbl_rows=-1):
    print(ovlp_df.group_by(['city', 'wave']).len('N').sort(['city', 'wave']))

shape: (15, 3)
┌───────────┬──────┬─────┐
│ city      ┆ wave ┆ N   │
│ ---       ┆ ---  ┆ --- │
│ str       ┆ i32  ┆ u32 │
╞═══════════╪══════╪═════╡
│ montreal  ┆ 1    ┆ 91  │
│ montreal  ┆ 2    ┆ 161 │
│ montreal  ┆ 3    ┆ 159 │
│ montreal  ┆ 4    ┆ 182 │
│ saskatoon ┆ 2    ┆ 76  │
│ saskatoon ┆ 3    ┆ 77  │
│ saskatoon ┆ 4    ┆ 41  │
│ vancouver ┆ 1    ┆ 131 │
│ vancouver ┆ 2    ┆ 92  │
│ vancouver ┆ 3    ┆ 75  │
│ vancouver ┆ 4    ┆ 82  │
│ victoria  ┆ 1    ┆ 147 │
│ victoria  ┆ 2    ┆ 139 │
│ victoria  ┆ 3    ┆ 116 │
│ victoria  ┆ 4    ┆ 81  │
└───────────┴──────┴─────┘


In [11]:
# ... and overlap descriptive stats
ovlp_df = ovlp_df.with_columns(overlap = (pl.max_horizontal('axl_tmin', 'gps_tmin') - pl.min_horizontal('axl_tmax', 'gps_tmax')).abs())
with pl.Config(tbl_rows=-1):
    print(ovlp_df.group_by(['city', 'wave']).agg(
        overlap_avg = pl.mean('overlap'),
        overlap_std = pl.std('overlap'),
        overlap_min = pl.min('overlap'),
        overlap_max = pl.max('overlap')
    ).sort(['city', 'wave']))

shape: (15, 6)
┌───────────┬──────┬───────────────────┬───────────────────┬───────────────────┬───────────────────┐
│ city      ┆ wave ┆ overlap_avg       ┆ overlap_std       ┆ overlap_min       ┆ overlap_max       │
│ ---       ┆ ---  ┆ ---               ┆ ---               ┆ ---               ┆ ---               │
│ str       ┆ i32  ┆ duration[μs]      ┆ duration[μs]      ┆ duration[μs]      ┆ duration[μs]      │
╞═══════════╪══════╪═══════════════════╪═══════════════════╪═══════════════════╪═══════════════════╡
│ montreal  ┆ 1    ┆ 23d 21h 24m 34s   ┆ 12d 5h 42m 56s    ┆ 3s 120ms          ┆ 90d 2h 41m 20s    │
│           ┆      ┆ 263219µs          ┆ 614618µs          ┆                   ┆ 30ms              │
│ montreal  ┆ 2    ┆ 26d 16h 54m 5s    ┆ 8d 2h 43m 42s     ┆ 3s 249ms          ┆ 29d 23h 59m 15s   │
│           ┆      ┆ 7913µs            ┆ 984108µs          ┆                   ┆ 375ms             │
│ montreal  ┆ 3    ┆ 27d 6h 2m 55s     ┆ 8d 20h 11m 30s    ┆ 175ms          